# 🔍 Semantic Search on Global Energy Data
### Built step by step — understanding every piece

**Dataset:** Global Energy Transition & Renewables (1900–2026) — OWID  
**Goal:** Search 4,459 country-year energy profiles using natural language  

---
## The core idea

Traditional search matches **keywords**. If you search "coal-dependent nations", it finds rows containing the word "coal-dependent" — and finds nothing, because that phrase doesn't exist in our data.

Semantic search matches **meaning**. The model understands that "coal-dependent", "heavy coal use", "high coal share" and "coal 70%" all point to the same concept.

**The challenge with this dataset:** It has 128 *numeric* columns — no natural text. So we first need to synthesize text documents from the numbers. This skill — turning structured data into searchable text — is one of the most valuable things you can learn.

---
## Step 1 — Install libraries

| Library | What it does |
|---|---|
| `sentence-transformers` | Downloads and runs the embedding model (MiniLM) |
| `chromadb` | Vector database: stores embeddings + lets us search them |
| `pandas` | Data manipulation |

In [2]:
!pip install sentence-transformers chromadb pandas --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 33.9 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 34.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 38.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 61.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that ar

---
## Step 2 — Load and inspect the data

In [10]:
import os
for folder in os.listdir('/kaggle/input'):
    print(f'/kaggle/input/{folder}/')
    for file in os.listdir(f'/kaggle/input/{folder}'):
        print(f'  └── {file}')

/kaggle/input/datasets/
  └── waddahali
  └── sherryheshmat


In [12]:
import pandas as pd
import numpy as np

df = pd.read_csv('/kaggle/input/datasets/waddahali/global-energy-transition-and-renewables-1900-2026/owid-energy-data.csv')

print(f"Shape: {df.shape}")
print(f"Years: {df['year'].min()} – {df['year'].max()}")
print(f"Entities: {df['country'].nunique()} (countries + regions)")

# Text columns vs numeric columns
text_cols = df.select_dtypes(include='object').columns.tolist()
num_cols  = df.select_dtypes(include='number').columns.tolist()
print(f"\nText columns ({len(text_cols)}): {text_cols}")
print(f"Numeric columns: {len(num_cols)}")
print()
print("👆 Notice: only country and iso_code are text.")
print("   We can't embed raw rows — we need to BUILD text from numbers.")

Shape: (23232, 130)
Years: 1900 – 2025
Entities: 314 (countries + regions)

Text columns (2): ['country', 'iso_code']
Numeric columns: 128

👆 Notice: only country and iso_code are text.
   We can't embed raw rows — we need to BUILD text from numbers.


---
## Step 3 — Document Synthesis

### Why do we need this?

An embedding model takes **text** as input and outputs a vector.  
Our data is **numbers** — so we convert each row into a descriptive sentence.

### Why add qualitative labels like "very high" and "low"?

Because the embedding model learned meaning from human language — not raw percentages.  
"renewables 71.0% (very high renewable share)" gives the model **two signals**:
- The precise number (71.0%)
- The human concept ("very high renewable share")

This means a query like **"clean energy leaders"** will find Norway even though the query contains no numbers.

### What makes a good document?
1. Include the most informative fields (shares, not just absolute values)
2. Add qualitative labels so concepts map to language
3. Skip fields with NaN — they add noise
4. Skip near-zero values — "solar 0.001%" is meaningless noise

In [13]:
def row_to_document(row):
    """
    Convert one row (country-year) into a rich text document.
    
    Design decisions:
    - We add qualitative labels ('very high', 'low') so the embedding model
      can link numeric values to human concepts.
    - We skip values below 0.1% — they're noise, not signal.
    - We use periods between facts (not commas) so the model treats
      each fact as a semi-independent statement.
    """
    parts = [f"{row['country']} energy profile {int(row['year'])}"]

    # Helper: label a percentage value
    def fossil_label(v):
        if v > 85: return 'very high fossil dependency'
        if v > 70: return 'high fossil dependency'
        if v > 50: return 'moderate fossil dependency'
        return 'low fossil dependency'

    def renew_label(v):
        if v > 40: return 'very high renewable share'
        if v > 20: return 'high renewable share'
        if v > 10: return 'growing renewable share'
        return 'low renewable share'

    if pd.notna(row.get('fossil_share_energy')):
        v = row['fossil_share_energy']
        parts.append(f"fossil fuels {v:.1f}% ({fossil_label(v)})")

    if pd.notna(row.get('coal_share_energy')) and row['coal_share_energy'] > 0.5:
        parts.append(f"coal {row['coal_share_energy']:.1f}%")

    if pd.notna(row.get('oil_share_energy')) and row['oil_share_energy'] > 0.5:
        parts.append(f"oil {row['oil_share_energy']:.1f}%")

    if pd.notna(row.get('gas_share_energy')) and row['gas_share_energy'] > 0.5:
        parts.append(f"gas {row['gas_share_energy']:.1f}%")

    if pd.notna(row.get('renewables_share_energy')):
        v = row['renewables_share_energy']
        parts.append(f"renewables {v:.1f}% ({renew_label(v)})")

    for src in ['solar', 'wind', 'nuclear', 'hydro']:
        col = f'{src}_share_energy'
        if pd.notna(row.get(col)) and row[col] > 0.1:
            parts.append(f"{src} {row[col]:.1f}%")

    if pd.notna(row.get('greenhouse_gas_emissions')):
        parts.append(f"greenhouse gas emissions {row['greenhouse_gas_emissions']:.1f} MtCO2")

    if pd.notna(row.get('primary_energy_consumption')):
        parts.append(f"total energy consumption {row['primary_energy_consumption']:.1f} TWh")

    if pd.notna(row.get('energy_per_capita')):
        parts.append(f"energy per capita {row['energy_per_capita']:.0f} kWh per person")

    return '. '.join(parts) + '.'


# Test it on 5 diverse countries
test_cases = [('Germany', 2023), ('Brazil', 2022), ('China', 2010),
              ('Norway', 2020), ('Saudi Arabia', 2023)]

for country, year in test_cases:
    row = df[(df['country'] == country) & (df['year'] == year)]
    if len(row):
        print(f"--- {country} {year} ---")
        print(row_to_document(row.iloc[0]))
        print()

--- Germany 2023 ---
Germany energy profile 2023. fossil fuels 75.8% (high fossil dependency). coal 15.5%. oil 37.1%. gas 23.2%. renewables 23.6% (high renewable share). solar 5.0%. wind 11.0%. nuclear 0.6%. hydro 1.5%. greenhouse gas emissions 183.9 MtCO2. total energy consumption 3178.9 TWh. energy per capita 37598 kWh per person.

--- Brazil 2022 ---
Brazil energy profile 2022. fossil fuels 51.4% (moderate fossil dependency). coal 4.5%. oil 38.1%. gas 8.8%. renewables 47.7% (very high renewable share). solar 2.0%. wind 5.5%. nuclear 1.0%. hydro 28.8%. greenhouse gas emissions 70.0 MtCO2. total energy consumption 3731.7 TWh. energy per capita 17744 kWh per person.

--- China 2010 ---
China energy profile 2010. fossil fuels 92.1% (very high fossil dependency). coal 70.3%. oil 18.0%. gas 3.8%. renewables 7.2% (low renewable share). wind 0.4%. nuclear 0.7%. hydro 6.4%. greenhouse gas emissions 3116.5 MtCO2. total energy consumption 29055.6 TWh. energy per capita 21498 kWh per person.

-

---
## Step 4 — Build the full corpus

We filter to rows that have enough data to be meaningful.  
We also capture **metadata** alongside each document — this lets us filter later  
(e.g. "only search rows after 2000", "only search real countries, not regions").

In [14]:
# Filter: real countries only (iso_code is a 3-letter code)
# and must have at least the core energy share columns
required_cols = ['coal_share_energy', 'renewables_share_energy', 'fossil_share_energy']
real_countries = df[
    df['iso_code'].notna() &
    (df['iso_code'].str.len() == 3) &
    df[required_cols].notna().all(axis=1)
].copy()

print(f"Usable rows: {len(real_countries)}")
print(f"Countries:   {real_countries['country'].nunique()}")
print(f"Year range:  {real_countries['year'].min()} – {real_countries['year'].max()}")

# Build documents and metadata
documents = []
metadatas = []
ids       = []

for idx, row in real_countries.iterrows():
    doc = row_to_document(row)
    if len(doc) < 50:  # skip empty-ish documents
        continue

    documents.append(doc)

    # Metadata = structured fields we can filter on later
    # Chroma requires all metadata values to be str, int, float, or bool
    meta = {
        'country':   str(row['country']),
        'year':      int(row['year']),
        'iso_code':  str(row['iso_code']),
        'fossil_share':    round(float(row['fossil_share_energy']), 1)   if pd.notna(row['fossil_share_energy'])    else -1.0,
        'renew_share':     round(float(row['renewables_share_energy']), 1) if pd.notna(row['renewables_share_energy']) else -1.0,
        'solar_share':     round(float(row['solar_share_energy']), 2)    if pd.notna(row['solar_share_energy'])    else -1.0,
        'wind_share':      round(float(row['wind_share_energy']), 2)     if pd.notna(row['wind_share_energy'])     else -1.0,
        'coal_share':      round(float(row['coal_share_energy']), 1)     if pd.notna(row['coal_share_energy'])     else -1.0,
    }
    metadatas.append(meta)
    ids.append(f"{row['iso_code']}_{int(row['year'])}")

print(f"\nDocuments built: {len(documents)}")
print(f"\nSample document:")
print(documents[0])
print(f"\nSample metadata:")
print(metadatas[0])

Usable rows: 4459
Countries:   79
Year range:  1965 – 2024

Documents built: 4459

Sample document:
Algeria energy profile 1965. fossil fuels 95.5% (very high fossil dependency). coal 3.3%. oil 62.2%. gas 30.0%. renewables 4.5% (low renewable share). hydro 4.5%. total energy consumption 24.8 TWh. energy per capita 2008 kWh per person.

Sample metadata:
{'country': 'Algeria', 'year': 1965, 'iso_code': 'DZA', 'fossil_share': 95.5, 'renew_share': 4.5, 'solar_share': 0.0, 'wind_share': 0.0, 'coal_share': 3.3}


---
## Step 5 — Load the embedding model

### What is an embedding model?

It's a neural network trained to map text → a list of numbers (a **vector**).  
The training objective: texts with similar meaning should produce vectors that are **close together** in space.

### Why `all-MiniLM-L6-v2`?

| Model | Dimensions | Size | Speed | Quality |
|---|---|---|---|---|
| `all-MiniLM-L6-v2` | 384 | 80MB | very fast | good |
| `all-mpnet-base-v2` | 768 | 420MB | slower | better |
| OpenAI `text-embedding-3-small` | 1536 | API call | fast | best |

For a Kaggle notebook with no GPU: MiniLM is the right default.  
The model runs **locally** — no API key needed.

### What does `.encode()` actually do?

It runs your text through the neural network and returns the final hidden state  
— a 384-dimensional vector that encodes the semantic content of the text.

In [15]:
from sentence_transformers import SentenceTransformer
import numpy as np

print("Loading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded.")

# --- UNDERSTANDING EMBEDDINGS ---
# Let's embed 4 phrases and look at what we get
test_phrases = [
    "countries heavily dependent on coal",
    "nations with high coal consumption",    # same meaning, different words
    "clean energy leaders with high renewables",
    "Saudi Arabia fossil fuel economy",       # very different meaning
]

test_vecs = model.encode(test_phrases)

print(f"\nEach phrase becomes a vector of shape: {test_vecs[0].shape}")
print(f"First 8 values of phrase 1: {test_vecs[0][:8].round(4)}")
print()

# Cosine similarity — the KEY concept
# Formula: similarity = dot(A, B) / (||A|| * ||B||)
# Result: 1.0 = identical meaning, 0.0 = unrelated, -1.0 = opposite
from numpy.linalg import norm

def cosine_sim(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

print("=== Cosine Similarity Between Phrases ===")
print(f"'coal dependent' vs 'high coal consumption':    {cosine_sim(test_vecs[0], test_vecs[1]):.4f}  ← SHOULD BE HIGH")
print(f"'coal dependent' vs 'clean energy leaders':     {cosine_sim(test_vecs[0], test_vecs[2]):.4f}  ← should be lower")
print(f"'coal dependent' vs 'Saudi Arabia fossil fuel': {cosine_sim(test_vecs[0], test_vecs[3]):.4f}  ← somewhere in between")
print()
print("This is the semantic magic: similar MEANING = high score, regardless of exact words.")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded.

Each phrase becomes a vector of shape: (384,)
First 8 values of phrase 1: [ 0.0351  0.015   0.0258  0.011   0.0531  0.0095  0.0032 -0.0093]

=== Cosine Similarity Between Phrases ===
'coal dependent' vs 'high coal consumption':    0.8087  ← SHOULD BE HIGH
'coal dependent' vs 'clean energy leaders':     0.2133  ← should be lower
'coal dependent' vs 'Saudi Arabia fossil fuel': 0.4073  ← somewhere in between

This is the semantic magic: similar MEANING = high score, regardless of exact words.


---
## Step 6 — Embed all documents

Now we run `.encode()` on all 4,459 documents.  
Each document becomes a 384-dimensional vector.

**Important:** We use `batch_size=64` — this processes 64 documents at a time  
instead of one-by-one, which is significantly faster.

In [16]:
print(f"Embedding {len(documents)} documents...")
print("(This is the slow step — done once, then saved to Chroma)\n")

embeddings = model.encode(
    documents,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"\nEmbeddings shape: {embeddings.shape}")
print(f"  {embeddings.shape[0]} documents")
print(f"  {embeddings.shape[1]} dimensions per document")
print(f"Memory used: ~{embeddings.nbytes / 1024 / 1024:.1f} MB")

# Let's look at two documents and their similarity
norway_idx  = next(i for i,m in enumerate(metadatas) if m['country']=='Norway'  and m['year']==2020)
germany_idx = next(i for i,m in enumerate(metadatas) if m['country']=='Germany' and m['year']==2023)
china_idx   = next(i for i,m in enumerate(metadatas) if m['country']=='China'   and m['year']==2010)

sim_nor_ger = cosine_sim(embeddings[norway_idx],  embeddings[germany_idx])
sim_nor_chn = cosine_sim(embeddings[norway_idx],  embeddings[china_idx])
sim_ger_chn = cosine_sim(embeddings[germany_idx], embeddings[china_idx])

print()
print("=== Sanity Check: Document Similarity ===")
print(f"Norway 2020   vs Germany 2023:  {sim_nor_ger:.4f}")
print(f"Norway 2020   vs China 2010:    {sim_nor_chn:.4f}")
print(f"Germany 2023  vs China 2010:    {sim_ger_chn:.4f}")
print()
print("Norway (71% renewables) should be LEAST similar to China 2010 (92% fossil).")
print("Germany (high fossil but transitioning) sits somewhere in between.")

Embedding 4459 documents...
(This is the slow step — done once, then saved to Chroma)



Batches:   0%|          | 0/70 [00:00<?, ?it/s]


Embeddings shape: (4459, 384)
  4459 documents
  384 dimensions per document
Memory used: ~6.5 MB

=== Sanity Check: Document Similarity ===
Norway 2020   vs Germany 2023:  0.6881
Norway 2020   vs China 2010:    0.5858
Germany 2023  vs China 2010:    0.6821

Norway (71% renewables) should be LEAST similar to China 2010 (92% fossil).
Germany (high fossil but transitioning) sits somewhere in between.


---
## Step 7 — Store in Chroma

### Why not just use a list?

You *could* compute cosine similarity between your query and every document manually.  
For 4,459 documents it would even be fast enough.

But Chroma gives you three things a plain list can't:

1. **Persistence** — embeddings saved to disk, not recomputed every time
2. **Metadata filtering** — `where={"year": {"$gte": 2010}}` runs server-side
3. **Scalability** — uses HNSW index (approximate nearest neighbour) that stays fast at millions of items

### What is HNSW?

Hierarchical Navigable Small World — a graph-based index structure.  
It builds a multi-layer graph of vectors so search navigates to the closest  
neighbours without comparing against every single vector.  
Think of it like a B-tree but for high-dimensional space.

In [17]:
import chromadb

# PersistentClient saves to disk — survives kernel restarts
# Use chromadb.Client() for pure in-memory (faster, no disk needed)
client = chromadb.PersistentClient(path="./energy_chroma_db")

# Delete collection if it already exists (clean restart)
try:
    client.delete_collection("energy_profiles")
    print("Deleted existing collection.")
except:
    pass

# Create collection
# cosine distance = 1 - cosine_similarity
# so closest = lowest distance = highest similarity
collection = client.create_collection(
    name="energy_profiles",
    metadata={"hnsw:space": "cosine"}
)

# Add in batches of 500 (Chroma's recommended batch size)
BATCH = 500
for i in range(0, len(documents), BATCH):
    batch_end = min(i + BATCH, len(documents))
    collection.add(
        documents  = documents[i:batch_end],
        embeddings = embeddings[i:batch_end].tolist(),
        metadatas  = metadatas[i:batch_end],
        ids        = ids[i:batch_end]
    )
    print(f"  Added {batch_end}/{len(documents)} documents")

print(f"\nCollection size: {collection.count()} documents")
print("Index built and saved to ./energy_chroma_db")

  Added 500/4459 documents
  Added 1000/4459 documents
  Added 1500/4459 documents
  Added 2000/4459 documents
  Added 2500/4459 documents
  Added 3000/4459 documents
  Added 3500/4459 documents
  Added 4000/4459 documents
  Added 4459/4459 documents

Collection size: 4459 documents
Index built and saved to ./energy_chroma_db


---
## Step 8 — The search function

### What happens when you search?

1. Your query string is passed through the **same embedding model**
2. You get a 384-dimensional query vector
3. Chroma finds the k document vectors with the **lowest cosine distance** to your query
4. Those documents are returned, ranked by similarity

**Critical:** Query and documents MUST use the same model.  
If you used `all-MiniLM-L6-v2` to embed the documents, you must use it for queries too.  
Different models produce vectors in different spaces — similarities would be meaningless.

### What do the distance scores mean?

Chroma returns **cosine distance** (not similarity):  
`distance = 1 - cosine_similarity`  
So: `0.0` = identical · `0.3` = strong match · `0.7` = weak match · `1.0` = opposites

In [18]:
def search(query, n_results=5, filters=None):
    """
    Search the energy corpus using natural language.

    Parameters
    ----------
    query     : plain English question or description
    n_results : how many results to return
    filters   : dict of Chroma metadata filters, e.g.
                {"year": {"$gte": 2010}}
                {"$and": [{"year": {"$gte": 2000}}, {"renew_share": {"$gte": 30}}]}

    Returns
    -------
    Prints ranked results with score and document text.
    """
    # Step 1: embed the query
    query_vec = model.encode([query], convert_to_numpy=True)

    # Step 2: search Chroma
    kwargs = dict(query_embeddings=query_vec.tolist(), n_results=n_results,
                  include=['documents', 'metadatas', 'distances'])
    if filters:
        kwargs['where'] = filters

    results = collection.query(**kwargs)

    # Step 3: display
    print(f"Query: \"{query}\"")
    if filters:
        print(f"Filter: {filters}")
    print("-" * 60)

    for i, (doc, meta, dist) in enumerate(
        zip(results['documents'][0], results['metadatas'][0], results['distances'][0])
    ):
        # Convert distance back to similarity (easier to read)
        similarity = round(1 - dist, 3)
        print(f"#{i+1}  {meta['country']} {meta['year']}  |  similarity: {similarity}")
        print(f"     {doc[:120]}...")
        print()

print("Search function ready.")

Search function ready.


---
## Step 9 — The wow queries

These are queries that **keyword search simply cannot answer**.  
Every single one works because the model understands *meaning*, not *words*.

In [19]:
# Query 1: Concept with no exact keywords in the data
search("countries rapidly transitioning away from fossil fuels")

Query: "countries rapidly transitioning away from fossil fuels"
------------------------------------------------------------
#1  Netherlands 1984  |  similarity: 0.464
     Netherlands energy profile 1984. fossil fuels 98.7% (very high fossil dependency). coal 9.3%. oil 43.8%. gas 45.6%. rene...

#2  Netherlands 1993  |  similarity: 0.462
     Netherlands energy profile 1993. fossil fuels 98.5% (very high fossil dependency). coal 9.9%. oil 45.1%. gas 43.5%. rene...

#3  Netherlands 1973  |  similarity: 0.461
     Netherlands energy profile 1973. fossil fuels 99.6% (very high fossil dependency). coal 4.1%. oil 56.8%. gas 38.8%. rene...

#4  Netherlands 1988  |  similarity: 0.457
     Netherlands energy profile 1988. fossil fuels 98.5% (very high fossil dependency). coal 10.9%. oil 47.3%. gas 40.3%. ren...

#5  Netherlands 1978  |  similarity: 0.456
     Netherlands energy profile 1978. fossil fuels 98.3% (very high fossil dependency). coal 4.7%. oil 50.6%. gas 43.0%. rene...



In [20]:
# Query 2: Uses synonym — 'petrostates' never appears in the data
search("petrostates with almost no renewable energy")

Query: "petrostates with almost no renewable energy"
------------------------------------------------------------
#1  Philippines 1969  |  similarity: 0.418
     Philippines energy profile 1969. fossil fuels 94.6% (very high fossil dependency). oil 94.4%. renewables 5.4% (low renew...

#2  Philippines 1971  |  similarity: 0.414
     Philippines energy profile 1971. fossil fuels 94.8% (very high fossil dependency). oil 94.6%. renewables 5.2% (low renew...

#3  Philippines 1972  |  similarity: 0.41
     Philippines energy profile 1972. fossil fuels 94.4% (very high fossil dependency). oil 94.2%. renewables 5.6% (low renew...

#4  Philippines 1974  |  similarity: 0.409
     Philippines energy profile 1974. fossil fuels 95.0% (very high fossil dependency). oil 94.7%. renewables 5.0% (low renew...

#5  Philippines 1970  |  similarity: 0.408
     Philippines energy profile 1970. fossil fuels 93.9% (very high fossil dependency). oil 93.7%. renewables 6.1% (low renew...



In [21]:
# Query 3: Conceptual — 'clean energy leaders' has no exact match
search("clean energy leaders in Europe", filters={"year": {"$gte": 2015}})

Query: "clean energy leaders in Europe"
Filter: {'year': {'$gte': 2015}}
------------------------------------------------------------
#1  Sweden 2024  |  similarity: 0.462
     Sweden energy profile 2024. fossil fuels 27.8% (low fossil dependency). coal 3.2%. oil 23.0%. gas 1.6%. renewables 51.3%...

#2  Sweden 2023  |  similarity: 0.458
     Sweden energy profile 2023. fossil fuels 27.1% (low fossil dependency). coal 3.2%. oil 22.4%. gas 1.5%. renewables 52.5%...

#3  Sweden 2020  |  similarity: 0.453
     Sweden energy profile 2020. fossil fuels 28.0% (low fossil dependency). coal 3.2%. oil 23.1%. gas 1.7%. renewables 51.1%...

#4  Sweden 2016  |  similarity: 0.451
     Sweden energy profile 2016. fossil fuels 32.3% (low fossil dependency). coal 3.8%. oil 26.8%. gas 1.7%. renewables 40.9%...

#5  Sweden 2022  |  similarity: 0.45
     Sweden energy profile 2022. fossil fuels 26.4% (low fossil dependency). coal 3.1%. oil 22.0%. gas 1.3%. renewables 52.4%...



In [22]:
# Query 4: Historical comparison — find profiles SIMILAR TO a reference
# Germany today = 75% fossil. Find who looked like that in the past.
search(
    "high fossil dependency but growing renewable energy sector",
    filters={"$and": [{"year": {"$gte": 2000}}, {"renew_share": {"$gte": 10}}]}
)

Query: "high fossil dependency but growing renewable energy sector"
Filter: {'$and': [{'year': {'$gte': 2000}}, {'renew_share': {'$gte': 10}}]}
------------------------------------------------------------
#1  United States 2024  |  similarity: 0.569
     United States energy profile 2024. fossil fuels 80.3% (high fossil dependency). coal 8.3%. oil 37.8%. gas 34.2%. renewab...

#2  United States 2023  |  similarity: 0.565
     United States energy profile 2023. fossil fuels 81.1% (high fossil dependency). coal 8.7%. oil 38.3%. gas 34.1%. renewab...

#3  United States 2022  |  similarity: 0.565
     United States energy profile 2022. fossil fuels 81.6% (high fossil dependency). coal 10.4%. oil 37.7%. gas 33.4%. renewa...

#4  China 2023  |  similarity: 0.563
     China energy profile 2023. fossil fuels 82.2% (high fossil dependency). coal 54.0%. oil 19.5%. gas 8.7%. renewables 15.5...

#5  Sri Lanka 2002  |  similarity: 0.562
     Sri Lanka energy profile 2002. fossil fuels 86.0% (very h

In [23]:
# Query 5: The 'solar boom' — no such phrase exists in the documents
search("countries with explosive solar growth", filters={"year": {"$gte": 2018}})

Query: "countries with explosive solar growth"
Filter: {'year': {'$gte': 2018}}
------------------------------------------------------------
#1  Iran 2024  |  similarity: 0.438
     Iran energy profile 2024. fossil fuels 98.1% (very high fossil dependency). coal 0.5%. oil 29.1%. gas 68.4%. renewables ...

#2  Iran 2022  |  similarity: 0.436
     Iran energy profile 2022. fossil fuels 98.4% (very high fossil dependency). coal 0.6%. oil 29.2%. gas 68.6%. renewables ...

#3  Iran 2023  |  similarity: 0.436
     Iran energy profile 2023. fossil fuels 97.9% (very high fossil dependency). coal 0.5%. oil 28.7%. gas 68.7%. renewables ...

#4  Uzbekistan 2023  |  similarity: 0.431
     Uzbekistan energy profile 2023. fossil fuels 96.9% (very high fossil dependency). coal 7.5%. oil 11.0%. gas 78.3%. renew...

#5  India 2022  |  similarity: 0.43
     India energy profile 2022. fossil fuels 89.4% (very high fossil dependency). coal 55.6%. oil 28.0%. gas 5.8%. renewables...



In [24]:
# Query 6: Free-form concept — 'energy poverty' is not in any column
search("low energy consumption developing nation", n_results=7)

Query: "low energy consumption developing nation"
------------------------------------------------------------
#1  Turkmenistan 2024  |  similarity: 0.546
     Turkmenistan energy profile 2024. fossil fuels 100.0% (very high fossil dependency). oil 23.1%. gas 76.8%. renewables 0....

#2  Turkmenistan 2022  |  similarity: 0.545
     Turkmenistan energy profile 2022. fossil fuels 100.0% (very high fossil dependency). oil 20.4%. gas 79.6%. renewables 0....

#3  Iran 2024  |  similarity: 0.544
     Iran energy profile 2024. fossil fuels 98.1% (very high fossil dependency). coal 0.5%. oil 29.1%. gas 68.4%. renewables ...

#4  Iran 2022  |  similarity: 0.54
     Iran energy profile 2022. fossil fuels 98.4% (very high fossil dependency). coal 0.6%. oil 29.2%. gas 68.6%. renewables ...

#5  Turkmenistan 1996  |  similarity: 0.538
     Turkmenistan energy profile 1996. fossil fuels 100.0% (very high fossil dependency). oil 24.1%. gas 75.5%. renewables 0....

#6  India 2024  |  similarity: 0.537

In [25]:
# Query 7: Anchor search — find countries similar to a specific reference
# What countries looked like China in 2010? (92% fossil, coal-dominated)
reference_doc = documents[china_idx]
print("Reference document (China 2010):")
print(reference_doc)
print()

# Embed the reference document itself as the query
ref_vec = model.encode([reference_doc], convert_to_numpy=True)
results = collection.query(
    query_embeddings=ref_vec.tolist(),
    n_results=6,
    include=['documents', 'metadatas', 'distances']
)

print("Countries most similar to China 2010:")
print("-" * 60)
for doc, meta, dist in zip(results['documents'][0], results['metadatas'][0], results['distances'][0]):
    if meta['country'] == 'China' and meta['year'] == 2010:
        continue  # skip the reference itself
    print(f"{meta['country']} {meta['year']}  |  similarity: {round(1-dist, 3)}")
    print(f"  {doc[:110]}...")
    print()

Reference document (China 2010):
China energy profile 2010. fossil fuels 92.1% (very high fossil dependency). coal 70.3%. oil 18.0%. gas 3.8%. renewables 7.2% (low renewable share). wind 0.4%. nuclear 0.7%. hydro 6.4%. greenhouse gas emissions 3116.5 MtCO2. total energy consumption 29055.6 TWh. energy per capita 21498 kWh per person.

Countries most similar to China 2010:
------------------------------------------------------------
China 2008  |  similarity: 0.974
  China energy profile 2008. fossil fuels 92.4% (very high fossil dependency). coal 72.3%. oil 17.0%. gas 3.2%. ...

China 2009  |  similarity: 0.974
  China energy profile 2009. fossil fuels 92.8% (very high fossil dependency). coal 72.5%. oil 16.9%. gas 3.3%. ...

China 2007  |  similarity: 0.962
  China energy profile 2007. fossil fuels 93.9% (very high fossil dependency). coal 73.8%. oil 17.3%. gas 2.8%. ...

China 2011  |  similarity: 0.961
  China energy profile 2011. fossil fuels 92.6% (very high fossil dependency). co

---
## What you built — and why it matters

| Skill demonstrated | Why it matters for AI Engineering |
|---|---|
| Document synthesis from numeric data | Most real datasets aren't text — this skill makes them searchable |
| Embedding model selection and usage | Core of every RAG, search, and recommendation system |
| Cosine similarity — intuition + code | The mathematical foundation of semantic search |
| Vector database (Chroma) | Standard tool in production AI pipelines |
| Metadata filtering + semantic search | What separates a toy from a real system |
| Anchor search (doc-to-doc similarity) | Enables "find me more like this" — used in recommendations |